# 13 · eval — **transfer** (짧은 task)

메인과 **완전히 같은 프로토콜**: 150k 체크포인트 × 5회 반복(rep 마다 env seed 변경) × 500 에피소드.
→ 모델당 4 seed × 5 rep = 20 run. 끝난 run 은 자동 skip.

`act_te` 를 넣으면 act 체크포인트에 eval-time TE 를 얹어 평가한다(학습 불필요).


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import importlib, common_final as cf
importlib.reload(cf)

TASK  = cf.SHORT_SIM       # ★ 'transfer' (AlohaTransferCube-v0) — 짧은 앵커
SEEDS = cf.MAIN_SEEDS      # [0,1,2,3]
GPUS  = cf.v23.available_gpus()
NGPU  = len(GPUS)

# ── 무엇을 평가할지 ───────────────────────────────────────────────────────
TAGS = cf.GROUP_OURS + cf.GROUP_ACM     # ['ours', 'acm'] — 핵심 비교 (8잡)
# TAGS = cf.GROUP_BASELINE              #   act·diffusion·smolvla·acm2 (16잡)
# TAGS = cf.GROUP_ABLATION              #   사다리 나머지 (12잡)
# TAGS = cf.TRAIN_ALL                   #   전부 (24잡)
REPS = list(range(cf.EVAL_REPEATS))
N_EP = cf.EVAL_N_EP

print('GPU :', GPUS, f'({NGPU}개)')
print('task:', TASK, '| ckpt', f'{cf.CKPT_STEP:,}', '| reps', REPS, '| n_ep', N_EP)
print('eval:', TAGS, '| 총 run:', len(TAGS) * len(SEEDS) * len(REPS))

## 사전 확인 — 150k 체크포인트 (transfer)

In [ ]:
ok = cf.print_ckpt_status([t for t in TAGS if t != 'act_te'], SEEDS, TASK)

## 반복 eval

In [ ]:
cf.run_repeat_evals(TAGS, SEEDS, REPS, task=TASK, ngpu=NGPU, n_episodes=N_EP)

## 결과 (SR) — transfer

In [ ]:
rows = cf.sr_table(TAGS, SEEDS, REPS, task=TASK, n_episodes=N_EP,
                   csv_path=cf.OUTPUT_BASE / 'main_report' / 'sr_150k_reps_transfer.csv')

## horizon 축 — transfer(짧) vs insertion(긺)
같은 모델의 두 task SR 을 나란히. **격차가 horizon 과 함께 커지는지**가 논문의 핵심 그림.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
                     'font.size': 13, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False})

TASKS_AXIS = [cf.SHORT_SIM, cf.MAIN_SIM]        # 짧 → 긺
out = cf.OUTPUT_BASE / 'main_report'
out.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 5))
found = False
for t in TAGS:
    ys, xs = [], []
    for i, tk in enumerate(TASKS_AXIS):
        agg = cf.sr_over_reps(t, task=tk, seeds=SEEDS, reps=REPS)
        if agg['mean'] is not None:
            xs.append(i); ys.append(agg['mean'])
    if len(ys) < 1:
        continue
    found = True
    ax.plot(xs, ys, '-o', ms=7, color=cf.COLOR.get(t, '#333'), label=cf.FINAL_LABELS.get(t, t))
if not found:
    print('아직 결과 없음 — 두 task 를 다 eval 한 뒤 다시 실행')
else:
    ax.set_xticks(range(len(TASKS_AXIS)))
    ax.set_xticklabels([f'{tk}\n(짧음)' if tk == cf.SHORT_SIM else f'{tk}\n(긴 horizon)'
                        for tk in TASKS_AXIS])
    ax.set_ylabel('Success rate (%)')
    ax.set_title('SR vs task horizon — 길수록 격차가 벌어지는가', fontweight='bold')
    ax.legend(fontsize=10)
    fig.savefig(out / 'sr_vs_horizon.png')
    fig.savefig(out / 'sr_vs_horizon.pdf')
    plt.show()
    print('저장:', out / 'sr_vs_horizon.png')